### Smart RAG Chatbot — Logging & Observability Setup

##### Overview
This document describes the logging and observability infrastructure for our Slack‑integrated RAG chatbot. The system uses **AWS Lambda** and **CloudWatch** to provide real‑time monitoring of chatbot performance.

---

### Architecture

##### System Flow
Slack Workspace → Slack Bot → AWS Lambda → Bedrock + Knowledge Base → Response → Slack

---

#### Components
- **Slack Integration**  
  Receives user questions via app mentions
- **Lambda Handler**  
  Thin orchestration layer (parses events, manages logging lifecycle)
- **RAG Pipeline**
  - `query_analyser.py` — Analyzes user intent
  - `retrieval_planner.py` — Plans retrieval strategy
  - `filter_gen.py` — Generates dynamic filters
  - `ask_smart.py` — Orchestrates pipeline and generates answers
- **CloudWatch**  
  Captures logs and metrics

---

#### Why CloudWatch?

CloudWatch enables:

- **Real‑time log ingestion**
- **Metrics tracking** (query count, latency, error rate)
- **Dashboard visualizations**
- **Free basic monitoring** with optional upgrades

---

### What We Log

#### Request Lifecycle

**When message arrives**
- `request_id`, `slack_user_id`, `channel`, `text`, `timestamp`

**After query analysis**
- Detected intent  
- Retrieval strategy  
- Complexity level  

**After Bedrock response**
- Confidence score  
- Citation count  
- Latency  
- Errors  

---

#### Log Categories

##### CloudWatch Logs
- Full query context  
- Stage‑by‑stage execution  
- Errors with stack traces  
- Component timings  

##### CloudWatch Metrics
- Query volume  
- Average latency  
- Confidence score distribution  
- Error and fallback rates  

---

#### Log Schema

##### Standard Fields (all logs)

```json
{
  "timestamp": "2024-01-15T10:30:45Z",
  "level": "INFO",
  "request_id": "uuid-here",
  "log_type": "component"
}

#### Log Types

1. Request Start

```
{
  "log_type": "request_start",
  "query": "What is RStudio?",
  "query_length_chars": 18,
  "environment": "lambda"
}
```

2. Component Execution

```
{
  "log_type": "component",
  "component_name": "query_analyser",
  "duration_ms": 245,
  "metadata": {
    "detected_intent": "definition",
    "complexity": "simple"
  }
}
```

3. Request Success

```

{
  "log_type": "request_success",
  "total_duration_ms": 3450,
  "complexity": "simple",
  "docs_retrieved": 5,
  "confidence": 0.92,
  "filters_used_summary": ["page_h1"],
  "answer_length_chars": 287
}

```
4. Request Error

```
{
  "log_type": "request_error",
  "failed_component": "filter_gen",
  "error_type": "ValidationError",
  "error_message": "Invalid filter structure",
  "stacktrace": "..."
}
```
---
#### Implementation Design

Logger Architecture

SmartRAGLogger (collects data)
       ↓
LogStorage Interface (writes data)
       ↓
CloudWatchBackend (formats & sends)

#### Lambda Integration Pattern

Lambda responsibilities:

  - Parse Slack event
  - Generate request_id
  - Initialize SmartRAGLogger with CloudWatchBackend
  - Call ask_smart() pipeline
  - Log final success/error
  - Return Slack-formatted response

ask_smart responsibilities:

  - Execute RAG pipeline logic
  - Call logger.log_component() for each stage
  - Return structured results
  - Remain AWS/Slack-agnostic

#### Execution Flow


```
Lambda invoked
  └── SmartRAGLogger(CloudWatchBackend) created
  └── ask_smart(query, logger)
      ├── logger.log_component("query_analyser", ...)
      ├── logger.log_component("retrieval_planner", ...)
      ├── logger.log_component("filter_gen", ...)
      └── logger.log_component("bedrock_retrieval", ...)
  └── logger.log_success() / logger.log_error()
  └── logger.flush() [auto-called]

```
---

#### AWS Setup Requirements

Lambda Permissions

Lambda execution role needs these CloudWatch permissions:
```
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": [
        "logs:CreateLogGroup",
        "logs:CreateLogStream",
        "logs:PutLogEvents"
      ],
      "Resource": "arn:aws:logs:*:*:*"
    }
  ]
}

```

What each does:

  - CreateLogGroup - Auto-creates log group on first run
  - CreateLogStream - Creates new stream for each execution context
  - PutLogEvents - Writes log entries
  
---

#### Environment Detection

The logger automatically detects execution environment:

  - Local (notebook): Pretty-printed console output for debugging
  - Lambda: Structured JSON to stdout (CloudWatch captures automatically)
Detection method: Check for AWS_LAMBDA_FUNCTION_NAME environment variable

---

#### Testing in This Notebook

Setup
  - Deploy Lambda function with deploy_lambda()
  - Verify CloudWatch permissions
  - Test with sample questions

**Test Function**

The test_question() function:

  - Invokes Lambda with Slack-formatted event
  - Displays answer and metadata
  - Fetches and parses CloudWatch logs
  - Shows execution trace with component timings

**What You'll See**

  - Answer: The chatbot's response
  - Response Time: Total Lambda execution duration
  - Request ID: For CloudWatch log tracking
  - Execution Trace: Component-by-component timing breakdown
  - Metrics: Confidence score, document count, etc.

---

#### Design Decisions

POC Scope

  - Sensitive Data: No PII masking needed (internal use case)
  - Log Volume: Errors + summary metrics (full logs available on-demand)
  - Filter Logging: Summary only (full JSON logged on errors)

**Lambda Deployment: Option A (Selected)**

Single Lambda Function

  - ✅ Simple deployment and management
  - ✅ Sufficient for POC workload
  - ⚠️ Risk of cold starts (acceptable for demo)
  - ⚠️ 15-minute timeout limit (queries complete in ~5s)

**Alternatives considered:**

  - Option B: Step Functions + multiple Lambdas (better observability, more complex)
  - Option C: Lambda + async SQS processing (handles high volume, adds latency)

---

#### Key Metrics to Monitor

**Performance**
  - QuestionCount by complexity (simple/complex)
  - ResponseTimeMs average and p95
  - QuestionsPerHour for usage trends

**Quality**
  - ErrorRate percentage
  - ZeroResultRate (indicates KB coverage gaps)
  - SupportReferralRate (fallback trigger frequency)

**Usage Patterns**
  - QuestionComplexityDistribution
  - FilterUsageCount by filter type
  - MostUsedFilters top 5
---

#### Next Steps

After this POC, consider:

Stress Testing: Simulate concurrent Slack users
Dashboard Creation: Visualize key metrics in CloudWatch
Alert Setup: Notify on error rate spikes or high latency
Migration Path: Plan for ECS deployment if scaling needed

#### Running the Demo

Execute cells below in sequence to:

  - Deploy Lambda
  - Test with multiple questions
  - Review logs and performance metrics


```text
SmartRAGLogger (collects data)
          ↓
   LogStorage Interface (writes data)
          ↓
   ┌───────────────┬───────────────┬
   ↓               ↓               ↓
CloudWatch       DynamoDB          S3
 (debug)     (conversations)   (analytics)


```text
SmartRag Lambda Flow
├── Lambda invoked
│     └── SmartRAGLogger(CloudWatchBackend) created
│
├── ask_smart(query, context, logger)
│     ├── logger.log_query_start()
│     ├── pipeline stages...
│     └── logger.log_query_complete()
│
└── logger.flush()



In [ ]:
# Implement basic Cloud watch logging
# Add Dynamo DB metrics storage
# set up X-ray tracing
# Cloud watch dashboards
# Configure alrets



## **The Flow:**

```
Postman (with token) 
    ↓
API Gateway (receives request)
    ↓
Lambda Authorizer (validates token) ← AUTH_TOKEN from .env
    ↓ (Allow/Deny decision)
    ↓
Your Main Lambda (processes query) ← Only if authorized
```

---

## **Why Lambda Authorizer validates:**

1. **Security Enforcement**
   - API Gateway doesn't know what valid tokens are
   - Lambda Authorizer has the AUTH_TOKEN from .env
   - It compares: `request token == AUTH_TOKEN`

2. **Without Authorizer:**
   - Anyone can call your API (public access)
   - No protection

3. **With Authorizer:**
   - Only requests with correct token reach your Lambda
   - Invalid tokens = blocked at API Gateway level
   - Your main Lambda never runs (saves cost + security)

---


In [ ]:
import os
from pathlib import Path
import sys

# Add project root to Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from helpers.apug.logging_observability.test_utils import test_question

# Setup
if Path.cwd().name == "notebooks":
    os.chdir('..')

from config import FUNCTION_NAME, REGION, AUTH_TOKEN
from deployment.deploy_lambda import deploy_lambda
from deployment.deploy_api_gateway import APIGatewayDeployer 
from deployment.create_lambda_layer import create_layer

In [ ]:
# Deploy Lambda (run once)
print("="*80)
print("DEPLOYING LAMBDA FUNCTION")
print("="*80)

create_layer()
deploy_lambda()

print("\n✅ Lambda deployment complete\n")

In [ ]:
# Deploy API Gateway using the class
deployer = APIGatewayDeployer()  # ← Create instance
endpoint_url = deployer.deploy()  # ← Call deploy()

print(f"\n✅ API Gateway deployed at: {endpoint_url}\n")

In [ ]:
# Terminal 1
#python app.py

# Terminal 2
#streamlit run streamlit_app.py

In [ ]:
# Test API
import requests
import json

response = requests.post(
    "{end_point}/ask",  # ← Use returned endpoint
    headers={
        "Content-Type": "application/json",
        "Authorization": f"Bearer {AUTH_TOKEN}"
    },
    json={"text": "What is R-studio?","verbose": True }
)

print(f"Status: {response.status_code}")
print(f"Response: {response.text}")




In [ ]:
""" In termianl you can try 

curl -X POST api endpoint \
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer token" \
  -d '{"text": "What is r-studio?", "verbose": true}'
  
  """

In [ ]:
""" 
Cloned cloud-platform-environments repo
Created folder: smartrag-chatbot-dev
"""

In [ ]:
"""
Phase 1: Observability Foundation  (Do First)
Why: Debug issues before adding complexity

X-Ray Tracing (1-2 hours)

Enables distributed tracing across Lambda → Bedrock → Knowledge Base
Shows exactly where time is spent
Critical for diagnosing latency/throttling
CloudWatch Metrics (30 min)

Error rate alarms
Latency monitoring
Free tier friendly


Phase 2: Data Collection  (Do Second)
Why: Start collecting real user data

DynamoDB (2-3 hours)
Store conversation history
Enable analytics queries
Fast lookup by request_id
Feedback Buttons (1-2 hours)
Simple thumbs up/down
Store in DynamoDB
Powers future improvements


Phase 3: Long-term Storage  (Do Last)
Why: Only needed at scale
S3 Export (1 hour)
Archive old conversations
Analytics/ML training data
Cost optimization (cheap storage)



Detailed Breakdown:
1. X-Ray Tracing (HIGHEST PRIORITY)
What you'll see:

API Gateway (5ms)
  └─ Lambda (1200ms)
      ├─ Query Analyzer (150ms)
      ├─ Bedrock KB Retrieve (800ms) ← Bottleneck!
      ├─ LLM Generation (250ms)
      └─ Response Format (5ms)   


"""

In [ ]:
#How does logging flow from SmartRAGLogger to CloudWatch in Lambda?
"""
Architecture Flow Diagram

SmartRAGLogger (your existing code - no changes needed)
    ↓ calls
CloudWatchBackend (update to use logging module)
    ↓ writes via
logging.getLogger('smartrag')
    ↓ captured by
Lambda → CloudWatch

 """

In [ ]:
# What are the different options for placing guardrails in the RAG pipeline?
"""    
## Three Options for Guardrails Placement

### **Option 1: Query Analysis Stage**
- Blocks silly questions **before** retrieval
-  Problem: Blocks internal analysis prompts
-  Saves retrieval costs

### **Option 2: Final Answer Stage** (I suggested)
- Blocks silly questions **after** retrieval
-  Internal operations work fine
-  Wastes retrieval on bad questions

### **Option 3: Separate Input Validation Call**
- Lightweight Bedrock call **before** query analysis
- Just ask: "Is this data engineering related? YES/NO"
-  Fast, cheap, doesn't interfere
-  Adds one small Bedrock call

### **Option 4: Hybrid - Both Places**
- Light validation at input (Option 3)
- Full guardrails at output (Option 2)
-  Best protection
-  Two Bedrock calls per request

---

# Which guardrails option was considered and why?

**Option 2** (Final answer only) because:
- Simplest implementation (one change)
- Catches everything (silly + harmful + PII)
- No risk of blocking legitimate queries
- Minimal cost waste


#######
Guardrails Architecture Flow
User question 
    ↓
Query Analysis (no guardrails)
    ↓
KB Retrieval
    ↓
Final Answer Generation
    ↓
Your prompt: "If not related to KB, say you can't answer"
    ↓
Guardrails: Block harmful/PII/off-topic
    ↓
User gets safe response


Why does the double protection (Prompt + Guardrails) work?
Prompt + Guardrails = Double protection
Prompt handles: "Not in KB knowledge"
Guardrails handle: "Silly/harmful questions"

"""

In [ ]:
""" 

Add feedback buttons (👍 👎) after each answer
Architecture Decision
User asks question
    ↓
Lambda generates answer
    ↓
Return answer + unique_message_id
    ↓
Show thumbs up/down buttons (Slack interactive components)
    ↓
User clicks → sends feedback to Lambda
    ↓
Store feedback (DynamoDB/S3/CloudWatch)

"""

In [ ]:
# Test 1
test_question("What is RStudio?")

In [ ]:
test_question("Good morning, all - I am unable to log into R Studio today (v4.5.1-1). I keep getting an error message that says" \
    "502 Bad Gateway" \
    "or sometimes" \
    "504 Gateway Time-out." \
        "Nothing i try is resolving this problem: restarting the laptop; refreshing the webpage; clearing the browser cache." \
            " Some helpful advice would be much appreciated!")

In [ ]:
test_question("Hello, Iam looking for some airflow help! We have a DAG in the DE Prod airflow which has been failing the last couple of days" \
      "but when I try to make a release of the fix using the existing DE github workflow the github action just sits in a queue for ages so the image" \
         " is not building. I thought the turn off was not until next month, has something else changed where we can not make new releases to the old airflow?")

In [ ]:
test_question("On the topic of the data uploader. Is there a process to request the removal of an existing table? " \
    "I have 2 tables that got data ingested and are now in Athena. But those tables changed during the latest designs review and now they are obsolete. " \
    "How can I get those deleted? What is the process to request such action? " \
    "Worth to highlight that the data stored in those tables is replaceable and there is no need to keep an archive of it")

In [ ]:
"""
What is the multi-layer security approach for guardrails?
Hybrid Guardrails Approach - Multi-Layer Security

Guardrails:

Hybrid Approach : System Prompt + Bedrock Guardrails
Layer 1: Input validation (regex, keywords)
    ↓
Layer 2: AWS Bedrock Guardrails
    ↓
Layer 3: Response sanitization (PII redaction)
    ↓
Layer 4: Output validation (length, format)

"""


In [ ]:
# AWS reference link:https://aws.amazon.com/blogs/machine-learning/create-a-generative-ai-assistant-with-slack-and-amazon-bedrock/

# The workflow consists of the following steps:
# 1. The user communicates with the Slack application
# 2. The Slack application send the even to Amazon API Gateway, which is used in the event subscription.
# 3. API Gateway forwards the event to an AWS Lambda function.
# 4. The Lambda function invokes Amazon Bedrock with the request, then responds to the user in Slack with the answer and sources.

In [ ]:
""" Priority 1: Must Do
1. Create DynamoDB table (run deployment/setup_dynamodb.py)
2. Update Lambda IAM permissions
3. Implement DynamoDBBackend.write_conversation()
4. Deploy and test with 10 queries
5. Verify data appears in DynamoDB
6. Add feedback endpoint  (Lambda + API Gateway route)
7. Streamlit feedback UI  (👍 👎 buttons)

Priority 2: Nice to Have
8. CloudWatch Dashboard (success rate, duration, ratings)
9. CloudWatch Logs Insights (pre-built queries)
10. Basic DynamoDB metrics monitoring
11. Document schema in README
12. Set up cost alerts

Priority 3: Can Wait
12. CloudWatch Alarms (error rate, latency thresholds)
13. Implement check_cache() placeholder
14. Add feature flag configuration
15. Plan advanced monitoring strategy (anomaly detection)

"""

In [ ]:
"""
What we've done:

✅ Created DynamoDB table (schema, GSIs, TTL)
✅ IAM permissions
✅ Implemented DynamoDBBackend.write_conversation() in log_backends.py
✅ Created dynamodb_query_utils.py (query/update helpers)
What we're doing next: We're NOT feeding the table TO Lambda - the table already receives data FROM Lambda (via DynamoDBBackend.write_conversation()).

What we're actually implementing:

Feedback endpoint (new Lambda function route) that uses dynamodb_query_utils.update_feedback() to UPDATE existing records
API Gateway route (POST /feedback) to expose this endpoint
Streamlit UI (👍👎 buttons) that calls the feedback endpoint
Flow:

User asks question → Lambda writes to DynamoDB ✅ (already working)
User clicks 👍/👎 → Streamlit calls /feedback → Lambda updates that record in DynamoDB ← this is what we're building

"""

In [ ]:
# ========== Rules ===========
 
# DynamoDB table setup (PK: request_id, SK: timestamp)
# UserTimeIndex + SessionIndex GSIs
# Lambda IAM permissions
# Backend implementation in DynamoDBBackend.write_conversation()

In [ ]:

# ==================== FAQ Cache Strategy ====================
# FUTURE ENHANCEMENT (Post-POC):
# 
# Goal: Cache high-confidence, frequently asked questions to reduce 
#       Bedrock API calls and improve response time
#
# Approach (Hybrid):
#   1. Log all requests to DynamoDB with confidence scores
#   2. Weekly analysis: Identify top 50 repeated questions (confidence >0.90)
#   3. Manual review & approval of candidates
#   4. Store approved FAQs in separate OpenSearch index ("rag-faq-cache")
#   5. Query flow: FAQ Cache (embedding similarity >0.95) → Primary KB → Log
#
# Implementation Steps:
#   - Create FAQ index in OpenSearch with query_embedding, answer, kb_version
#   - Add check_faq_cache() before primary KB retrieval
#   - Build weekly analysis job (Lambda + EventBridge)
#   - Create approval dashboard (Streamlit admin page)
#
# Benefits:
#   - Faster responses for common queries (<500ms vs 2-5s)
#   - Reduced Bedrock costs
#   - Self-improving system
#
# Note: Requires 100+ logged conversations before implementing
# ============================================================
